# 15 — FOMC decisions and closing-auction volume

Does the closing auction behave differently on days the Federal Reserve announces a rate decision?
Self-contained; nothing here modifies the panel or any earlier notebook.

### Why this is a small study, not a big one

Options gave every stock its own expiring open interest, so there were millions of effectively
independent observations. **FOMC does not.** Every stock experiences the same meeting on the same
day, so 2,250 trading days contain only about **70 FOMC events**. A statistic computed over three
million stock-days would look far more significant than the evidence warrants — it is one
observation seen 1,300 times.

Everything below therefore works at the **day level**: collapse each date to the median auction
residual across stocks first, then compare dates. Significance is assessed by permutation against
randomly chosen days, which respects the real sample size. Stock-level cuts appear only where the
question is genuinely cross-sectional (does the effect differ by size), and are read cautiously.

### What the data can and cannot be used for

The *date* of a scheduled meeting is published about a year ahead, so an `is_fomc_day` flag is known
in advance and is legitimate for the daily model, like `monthly_options_expiry`. The *decision* —
hike, cut, hold, basis points, vote split — is released at **2:00 p.m. ET**, two hours before the
close. That is not available at the prior close, so it cannot enter the daily model at all; it would
only be available to the 15:00 and 15:30 intraday checkpoints, which have not been built.

So section 4 onward describes behaviour and sizes the opportunity for a future intraday model. Only
section 3 tests something the daily model could actually use today.

Coverage: FOMC data begins 2016-11, so this notebook runs **2017 onwards** and drops two of the
eleven target years. The two unscheduled 2020 meetings are flagged and excluded from the
known-in-advance tests.

In [ ]:
from pathlib import Path
import warnings

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 120})

NAVY, GOLD, GREY = "#020225", "#D2BB6F", "#8A8A93"

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "blackrock_data").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate blackrock_data. Run this inside the project.")

DATA = PROJECT_ROOT / "blackrock_data"
OUT = PROJECT_ROOT / "outputs" / "eda" / "15_fomc_auction"
OUT.mkdir(parents=True, exist_ok=True)

PANEL = PROJECT_ROOT / "outputs" / "eda" / "05_feature_engineering" / "enriched_daily_panel.parquet"
FOMC = DATA / "events" / "fomc_decisions.csv"
JOINT = OUT / "fomc_auction_daily.parquet"

for p in (PANEL, FOMC):
    if not p.exists():
        raise FileNotFoundError(f"{p} missing.")

con = duckdb.connect()
con.execute("PRAGMA threads=4")

def sql(q):
    return con.execute(q).df()

FINDINGS = []
def note(label, value):
    FINDINGS.append({"finding": label, "value": value})
    print(f"  * {label}: {value}")

START_YEAR = 2017
print(f"Project root: {PROJECT_ROOT}")
print(f"Study window: {START_YEAR}-2025")

## 1. Build the day-level table

Two steps. First collapse the panel to one row per trading day — the median auction residual across
all eligible stocks, plus a few day-level descriptors. Then attach the FOMC decision for dates that
have one.

`auction_vs_normal` is the auction divided by that stock's own 20-day median, so 1.0 is an ordinary
day for that stock. Taking the median across stocks gives a clean market-wide reading per date that
is not dominated by the largest names.

In [ ]:
daily_sql = f"""
WITH s AS (
    SELECT CAST(date AS DATE) AS date,
           auction_volume / NULLIF(median20_auction_volume, 0) AS auction_vs_normal,
           auction_volume,
           monthly_options_expiry, quarterly_options_expiry, is_last_5_trading_days_month
    FROM read_parquet('{PANEL}')
    WHERE eligible_target AND median20_auction_volume > 0 AND year >= {START_YEAR}
),
d AS (
    SELECT date,
           count(*)                        AS n_stocks,
           median(auction_vs_normal)       AS day_median,
           avg(ln(auction_vs_normal))      AS day_mean_log,
           quantile_cont(auction_vs_normal, 0.90) AS day_p90,
           sum(auction_volume)             AS total_auction_shares,
           max(CAST(monthly_options_expiry AS INT))       AS monthly_expiry,
           max(CAST(quarterly_options_expiry AS INT))     AS quarterly_expiry,
           max(CAST(is_last_5_trading_days_month AS INT)) AS month_end_window
    FROM s WHERE auction_vs_normal > 0 GROUP BY 1
),
f AS (
    SELECT CAST(announcement_date AS DATE) AS date,
           decision, change_bp, new_upper_pct, votes_against, votes_cast,
           meeting_type, dissent_category
    FROM read_csv_auto('{FOMC}')
)
SELECT d.*,
       f.decision, f.change_bp, f.new_upper_pct, f.votes_against, f.votes_cast,
       f.meeting_type, f.dissent_category,
       CASE WHEN f.decision IS NULL THEN 0 ELSE 1 END          AS is_fomc_day,
       CASE WHEN COALESCE(f.change_bp, 0) <> 0 THEN 1 ELSE 0 END AS rate_changed,
       CASE WHEN COALESCE(f.votes_against, 0) > 0 THEN 1 ELSE 0 END AS has_dissent,
       abs(COALESCE(f.change_bp, 0))                            AS abs_change_bp
FROM d LEFT JOIN f USING (date)
ORDER BY d.date
"""
con.execute(f"COPY ({daily_sql}) TO '{JOINT}' (FORMAT PARQUET)")

daily = sql(f"SELECT * FROM read_parquet('{JOINT}') ORDER BY date")
print(f"Trading days: {len(daily):,}   FOMC days matched: {int(daily['is_fomc_day'].sum())}")
note("Trading days in window", f"{len(daily):,}")
note("FOMC events matched to a trading day", int(daily["is_fomc_day"].sum()))
daily.head()

## 2. Did every announcement land on a trading day?

An FOMC date that does not appear in the panel is a silent loss. This checks the CSV against what
actually matched, and reports anything unmatched.

In [ ]:
fomc_all = sql(f"""
    SELECT CAST(announcement_date AS DATE) AS date, decision, meeting_type
    FROM read_csv_auto('{FOMC}')
    WHERE CAST(announcement_date AS DATE) >= DATE '{START_YEAR}-01-01'
      AND CAST(announcement_date AS DATE) <= DATE '2025-12-31'
""")
matched = set(daily.loc[daily["is_fomc_day"] == 1, "date"])
fomc_all["matched"] = fomc_all["date"].isin(matched)

print(f"FOMC announcements {START_YEAR}-2025: {len(fomc_all)}")
print(f"Matched to a trading day in the panel: {int(fomc_all['matched'].sum())}")
unmatched = fomc_all[~fomc_all["matched"]]
if len(unmatched):
    print("\nUNMATCHED - investigate before trusting the results:")
    display(unmatched)
else:
    print("All announcements matched.")

by_year = (daily[daily["is_fomc_day"] == 1]
           .assign(year=lambda d: pd.to_datetime(d["date"]).dt.year)
           .groupby("year").size().rename("fomc_days").reset_index())
by_year.to_csv(OUT / "fomc_days_by_year.csv", index=False)
by_year.T

## 3. Are auctions bigger on FOMC days?

The headline question, and the only one the daily model could act on today, since the meeting date
is known a year ahead.

Compared at the day level: each date contributes one number. The permutation test draws the same
number of random trading days 5,000 times and asks how often a gap this large appears by chance —
the honest way to judge roughly 70 events.

In [ ]:
fomc = daily[daily["is_fomc_day"] == 1]
other = daily[daily["is_fomc_day"] == 0]

summary = pd.DataFrame({
    "group": ["FOMC day", "ordinary day"],
    "n_days": [len(fomc), len(other)],
    "median_of_day_medians": [fomc["day_median"].median(), other["day_median"].median()],
    "mean_of_day_medians": [fomc["day_median"].mean(), other["day_median"].mean()],
    "median_day_p90": [fomc["day_p90"].median(), other["day_p90"].median()],
})
summary.to_csv(OUT / "fomc_vs_ordinary_days.csv", index=False)
display(summary)

obs = fomc["day_median"].median() - other["day_median"].median()
pool = daily["day_median"].to_numpy()
k = len(fomc)
rng = np.random.default_rng(0)
null = np.empty(5000)
for i in range(5000):
    idx = rng.choice(len(pool), k, replace=False)
    mask = np.zeros(len(pool), dtype=bool); mask[idx] = True
    null[i] = np.median(pool[mask]) - np.median(pool[~mask])
p_value = float((np.abs(null) >= abs(obs)).mean())

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.hist(null, bins=60, color=GREY, label="random days (5,000 draws)")
ax.axvline(obs, color=NAVY, lw=2.5, label=f"observed FOMC gap = {obs:+.4f}")
ax.set_xlabel("difference in median auction residual")
ax.set_title(f"Permutation test on {k} FOMC days   (p = {p_value:.3f})")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "fomc_permutation_test.png", bbox_inches="tight")
plt.show()

note("FOMC vs ordinary day, median gap", round(float(obs), 4))
note("Permutation p-value", round(p_value, 4))
pd.DataFrame([{"observed_gap": obs, "p_value": p_value, "n_fomc_days": k,
               "null_sd": float(null.std())}]).to_csv(
    OUT / "fomc_permutation_test.csv", index=False)

## 4. Hikes, cuts and holds

Does the kind of decision matter? Counts are small — roughly 50 holds, 20 hikes and 11 cuts across
the full file, fewer inside this window — so treat anything here as indicative. Group medians on
fewer than ten days are reported but should not be relied on.

From here on the analysis describes behaviour rather than testing a usable daily feature: the
decision is not known until 2:00 p.m.

In [ ]:
by_decision = (daily[daily["is_fomc_day"] == 1]
               .groupby("decision")
               .agg(n_days=("day_median", "size"),
                    median_day_median=("day_median", "median"),
                    mean_day_median=("day_median", "mean"),
                    median_day_p90=("day_p90", "median"))
               .reset_index()
               .sort_values("median_day_median", ascending=False))
by_decision["reliable"] = np.where(by_decision["n_days"] >= 10, "yes", "too few days")
by_decision.to_csv(OUT / "fomc_by_decision.csv", index=False)
display(by_decision)

extra = (daily[daily["is_fomc_day"] == 1]
         .assign(group=lambda d: np.where(d["rate_changed"] == 1, "rate changed", "rate held"))
         .groupby("group")
         .agg(n_days=("day_median", "size"), median_day_median=("day_median", "median"))
         .reset_index())
dissent = (daily[daily["is_fomc_day"] == 1]
           .assign(group=lambda d: np.where(d["has_dissent"] == 1, "with dissent", "unanimous"))
           .groupby("group")
           .agg(n_days=("day_median", "size"), median_day_median=("day_median", "median"))
           .reset_index())
contrasts = pd.concat([extra, dissent], ignore_index=True)
contrasts.to_csv(OUT / "fomc_contrasts.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(by_decision["decision"], by_decision["median_day_median"], color=NAVY)
axes[0].axhline(other["day_median"].median(), color=GOLD, ls="--", lw=1.6, label="ordinary day")
axes[0].set_ylabel("median auction / own normal")
axes[0].set_title("By decision type")
axes[0].legend(fontsize=8)
axes[1].bar(contrasts["group"], contrasts["median_day_median"], color=[NAVY, NAVY, GREY, GREY])
axes[1].axhline(other["day_median"].median(), color=GOLD, ls="--", lw=1.6)
axes[1].set_title("Rate change and dissent")
plt.setp(axes[1].get_xticklabels(), rotation=15, ha="right")
fig.tight_layout()
fig.savefig(OUT / "fomc_by_decision.png", bbox_inches="tight")
plt.show()
contrasts

## 5. The days around an announcement

Markets often quieten before a decision and settle after it. Offsets are in trading days relative to
each FOMC announcement, market-wide.

In [ ]:
d = daily.reset_index(drop=True).copy()
d["t"] = np.arange(len(d))
event_t = d.loc[d["is_fomc_day"] == 1, "t"].to_numpy()

rows = []
for off in range(-5, 6):
    idx = event_t + off
    idx = idx[(idx >= 0) & (idx < len(d))]
    sel = d.iloc[idx]
    rows.append({"offset_days": off, "n_days": len(sel),
                 "median_day_median": sel["day_median"].median()})
event = pd.DataFrame(rows)
event.to_csv(OUT / "fomc_event_study.csv", index=False)

fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.plot(event["offset_days"], event["median_day_median"], marker="o", color=NAVY)
ax.axvline(0, color=GOLD, lw=2.5, alpha=.85)
ax.axhline(other["day_median"].median(), color="grey", ls="--", lw=1,
           label="ordinary-day level")
ax.set_xlabel("trading days relative to the FOMC announcement")
ax.set_ylabel("median auction / own normal")
ax.set_title("Closing-auction volume around FOMC decisions")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "fomc_event_study.png", bbox_inches="tight")
plt.show()
event

## 6. Is it confounded with the days we already know are busy?

Quarterly expiry lifts the auction to roughly 6.5x normal and month-end is also heavy. If FOMC dates
happen to coincide with those, an apparent FOMC effect would really be theirs. This checks the
overlap and then recomputes the comparison on clean days only.

In [ ]:
overlap = pd.crosstab(daily["is_fomc_day"],
                      daily[["monthly_expiry", "quarterly_expiry", "month_end_window"]].sum(axis=1) > 0,
                      rownames=["is_fomc_day"], colnames=["busy_calendar_day"])
display(overlap)

clean = daily[(daily["monthly_expiry"] == 0) & (daily["quarterly_expiry"] == 0)
              & (daily["month_end_window"] == 0)]
clean_cmp = pd.DataFrame({
    "group": ["FOMC day", "ordinary day"],
    "n_days": [int((clean["is_fomc_day"] == 1).sum()), int((clean["is_fomc_day"] == 0).sum())],
    "median_day_median": [clean.loc[clean["is_fomc_day"] == 1, "day_median"].median(),
                          clean.loc[clean["is_fomc_day"] == 0, "day_median"].median()],
})
clean_cmp.to_csv(OUT / "fomc_excluding_busy_days.csv", index=False)
gap_clean = clean_cmp["median_day_median"][0] - clean_cmp["median_day_median"][1]
note("FOMC gap excluding expiry and month-end days", round(float(gap_clean), 4))
clean_cmp

## 7. Does it differ by stock size?

The only genuinely cross-sectional question here. Within a single FOMC day all stocks share the
event, but whether large and small names respond differently is a real comparison. Still only about
70 days of evidence, so read the shape rather than the exact numbers.

In [ ]:
fomc_dates = daily.loc[daily["is_fomc_day"] == 1, "date"].tolist()
dates_sql = ", ".join(f"DATE '{pd.Timestamp(x).date()}'" for x in fomc_dates)

by_size = sql(f"""
SELECT size_decile,
       count(*) AS stock_days,
       median(CASE WHEN CAST(date AS DATE) IN ({dates_sql})
                   THEN auction_volume / NULLIF(median20_auction_volume, 0) END) AS fomc_day,
       median(CASE WHEN CAST(date AS DATE) NOT IN ({dates_sql})
                   THEN auction_volume / NULLIF(median20_auction_volume, 0) END) AS ordinary_day
FROM read_parquet('{PANEL}')
WHERE eligible_target AND median20_auction_volume > 0 AND year >= {START_YEAR}
  AND size_decile IS NOT NULL
GROUP BY 1 ORDER BY 1
""")
by_size["lift"] = by_size["fomc_day"] - by_size["ordinary_day"]
by_size.to_csv(OUT / "fomc_lift_by_size_decile.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(by_size["size_decile"], by_size["lift"], color=NAVY)
ax.axhline(0, color="grey", lw=1)
ax.set_xlabel("size decile (1 = smallest)")
ax.set_ylabel("FOMC-day auction lift")
ax.set_title("Does the FOMC effect depend on stock size?")
fig.tight_layout()
fig.savefig(OUT / "fomc_lift_by_size_decile.png", bbox_inches="tight")
plt.show()
by_size

## 8. Findings

### How to read this

**Section 3 is the only test of something usable today.** The meeting date is known a year ahead, so
`is_fomc_day` could be added to the daily model as a calendar feature. The permutation p-value is
the number that matters — with roughly 70 events, a gap needs to be sizeable to mean anything. Treat
p above 0.05 as no evidence, and judge the magnitude too: a statistically detectable gap of 0.01 on
a residual that is routinely 1.5x on expiry days is not worth a feature.

**Section 6 is the sanity check.** If the gap disappears once expiry and month-end days are removed,
the effect was theirs.

**Sections 4, 5 and 7 are descriptive.** The decision itself arrives at 2:00 p.m. and cannot enter
the daily model. Their value is sizing a future intraday opportunity, where the 15:00 and 15:30
checkpoints would see the announcement.

### What this data cannot support

With about 70 events there is no basis for modelling vote splits or dissent categories — 23 dissents
across five categories is a handful of days per cell. The decision-type breakdown in section 4 is
already near the limit.

More importantly, what should matter is not the decision but the **surprise** relative to what the
market expected, and nothing here measures that. A widely anticipated hold and a shock hike look
identical in these columns. Fed funds futures or OIS-implied expectations would be needed, and are
not in the repository.

### If section 3 shows something

Add `is_fomc_day` to notebook 05 as a calendar feature and re-test against the frozen benchmark in
`outputs/eda/09_global_finalists/frozen_global_benchmark.json`, the same way options were tested.
It is one column, so the bar is low — but so is the expected gain.

### If it does not

The FOMC layer stays a description of market behaviour, and the decision columns wait for the
intraday model, which is where a 2:00 p.m. release can actually be used.

In [ ]:
findings = pd.DataFrame(FINDINGS)
findings.to_csv(OUT / "fomc_auction_findings.csv", index=False)

artifacts = sorted(p.name for p in OUT.iterdir() if p.suffix in {".csv", ".png", ".parquet"})
print(f"{len(artifacts)} artifacts in {OUT}\n")
for a in artifacts:
    print("   ", a)
print()
findings